### Script for aggregating up the job accessability LAD data up to ITL 2 and 3
##### This script uses the uknownward 'network effects' data avaialbe at: https://github.com/ukonward/network_effects
##### Please note - This data is based on the 2021 ITL boundaries. 

Variable list:

* LSOA_code
* LAD_code
* LAD_name
* MSOA_code
* POP2019
* BRESEMP19
* COUNTRY
* RGN11CD
* RGN11NM
* DRIVE15 - job count within 15-minute drive
* DRIVE30 - job count within 30-minute drive
* DRIVE60 - job count within 60-minute drive
* DRIVE90 - job count within 90-minute drive
* PTRANS15 - job count within 15-minutes by public transport
* PTRANS30 - job count within 30-minutes by public transport
* PTRANS60 - job count within 60-minutes by public transport
* PTRANS90 - job count within 90-minutes by public transport
* AREAEHECT - total area of LSOA
* DRIVE15SQKM - total area reachable within 15-minute drive
* DRIVE30SQKM - total area reachable within 30-minute drive
* DRIVE60SQKM - total area reachable within 60-minute drive
* DRIVE90SQKM - total area reachable within 90-minute drive
* PTRANS15SQKM - total area reachable within 15-minutes by public transport
* PTRANS30SQKM - total area reachable within 30-minutes by public transport
* PTRANS60SQKM - total area reachable within 60-minutes by public transport
* PTRANS90SQKM - total area reachable within 90-minutes by public transport
* D15_jobspersqkm - total reachable jobs divided by total reachable area (DRIVE15 / DRIVE15SQKM)
* D30_jobspersqkm - (DRIVE30 / DRIVE30SQKM)
* D60_jobspersqkm - (DRIVE60 / DRIVE60SQKM)
* D90_jobspersqkm - (DRIVE90 / DRIVE90SQKM)
* PT15_jobspersqkm - (PTRANS15 / PTRANS15SQKM)
* PT30_jobspersqkm - (PTRANS30 / PTRANS30SQKM)
* PT60_jobspersqkm - (PTRANS60 / PTRANS60SQKM)
* PT90_jobspersqkm - (PTRANS90 / PTRANS90SQKM)
* 3mile - job count within 3 mile radius
* 5mile - job count within 5 mile radius
* 10mile - job count within 10 mile radius
* 35mile - job count within 35 mile radius
* ratio_5m_pt60 - ratio of PTRANS60 to 3mile
* IMD19
* SIMD20

In [2]:
# docustring and imports

"""
Aggregate LSOA-level job-access data up to ITL regions using a LAD -> ITL look-up.
Usage:

    python aggregate_to_itl.py                  # ITL3 (default)

    python aggregate_to_itl.py --level 2        # ITL2

    python aggregate_to_itl.py --split-lads drop
"""
import argparse
import pandas as pd

In [ ]:
# Variable Creation

DATA_PATH = "../lib"
LOOKUP_PATH = "../raw/job_accessability"

DATA_LAD_COL = "LAD_code"        # LAD code column in the data
LOOKUP_LAD_COL = "LAD20CD"       # LAD code column in the look-up
WEIGHT_COL = "POP2019"           # weights for averaged measures

# Columns that are genuine totals and can safely be summed
SUM_COLS = ["POP2019", "BRESEMP19", "AREAEHECT"]

# Identifier / descriptive columns that make no sense at ITL level
DROP_COLS = ["LSOA_code", "LAD_code", "LAD_name", "MSOA_code",

             "COUNTRY", "RGN11CD", "RGN11NM"]

# Every other numeric column (jobs reachable, area reachable, jobs per sq km,
# jobs within X miles, ratio, IMD/SIMD rank, access score) is aggregated as a
# population-weighted mean, because summing them would double-count.

In [3]:
# load look up function
def load_lookup(path, level, split_policy):

    """Return a one-row-per-LAD look-up with ITL code and name at `level`."""
    code_col, name_col = f"ITL{level}21CD", f"ITL{level}21NM"
    lk = pd.read_csv(path, dtype=str)[[LOOKUP_LAD_COL, code_col, name_col]]
    lk = lk.drop_duplicates()

    # The look-up is really at LAU1 level, so a few Scottish LADs are split
    # across several ITL regions. At the chosen level, find those that remain
    # ambiguous (more than one ITL code for a single LAD).

    n_codes = lk.groupby(LOOKUP_LAD_COL)[code_col].nunique()
    split = n_codes[n_codes > 1].index.tolist()
    if split:
        print(f"WARNING: {len(split)} LAD(s) fall in more than one ITL{level} "
              f"region and can't be split at LAD level: {split}")
        if split_policy == "drop":
            print("         -> dropping them from the output.")
            lk = lk[~lk[LOOKUP_LAD_COL].isin(split)]
        else:
            print("         -> assigning each to the first ITL region listed "
                  "in the look-up.")

    lk = lk.drop_duplicates(subset=LOOKUP_LAD_COL, keep="first")
    return lk.rename(columns={LOOKUP_LAD_COL: DATA_LAD_COL}), code_col, name_col

In [4]:
# weighted mean function

def weighted_mean(df, group_cols, value_cols, weight_col):
    """Weighted mean per group, ignoring NaNs in each column separately."""
    w = df[weight_col]
    out = {}
    for c in value_cols:
        valid = df[c].notna()
        num = (df[c] * w).where(valid).groupby([df[g] for g in group_cols]).sum()
        den = w.where(valid).groupby([df[g] for g in group_cols]).sum()
        mean = num / den
        # all-NaN groups stay NaN; zero-weight groups fall back to a simple mean
        simple = df.groupby(group_cols)[c].mean()
        out[c] = mean.where(den > 0, simple)
    return pd.DataFrame(out)

In [5]:
# main file splitting function

def main(level, split_policy, out_path):
    data = pd.read_csv(DATA_PATH, low_memory=False)
    lookup, code_col, name_col = load_lookup(LOOKUP_PATH, level, split_policy)

    # Some columns contain Excel errors such as '#DIV/0!' -> turn into NaN
    value_cols = [c for c in data.columns if c not in DROP_COLS]
    for c in value_cols:
        data[c] = pd.to_numeric(data[c], errors="coerce")
# Join LAD -> ITL
    merged = data.merge(lookup, on=DATA_LAD_COL, how="left", validate="m:1")
    unmatched = merged[merged[code_col].isna()]
    if len(unmatched):
        print(f"WARNING: {len(unmatched)} rows ({unmatched[DATA_LAD_COL].nunique()} "
              f"LADs) have no ITL match and are excluded: "
              f"{sorted(unmatched[DATA_LAD_COL].unique())}")
    merged = merged.dropna(subset=[code_col])

    group = [code_col, name_col]
    mean_cols = [c for c in value_cols if c not in SUM_COLS]

    # Sums (min_count=1 keeps all-NaN groups as NaN, e.g. AREAEHECT for Scotland)
    sums = merged.groupby(group)[SUM_COLS].sum(min_count=1)
    means = weighted_mean(merged, group, mean_cols, WEIGHT_COL)
    counts = merged.groupby(group).size().rename("n_LSOAs")

    result = pd.concat([counts, sums, means], axis=1).reset_index()
    result.to_csv(out_path, index=False)
    print(f"Aggregated {len(merged):,} LSOAs into {len(result)} ITL{level} "

          f"regions -> {out_path}")

    return result

In [ ]:
# Output
if __name__ == "__main__":
    SPLIT_POLICY = "first"  # "first" or "drop" for LADs split across ITL regions
    OUT_DIR = "../clean/"

    results = {}
    for level in (2, 3):
        out_path = f"{OUT_DIR}/job_access_itl{level}.csv"
        results[level] = main(level, SPLIT_POLICY, out_path)
    print(results[2].head())